# 🔐 SigAuth Complete Backend - Production Ready

## Advanced Forgery-Resilient Signature Authentication
### Using Siamese Metric Learning and Digital Tamper Detection

---

**Features:**
- Siamese Network with ResNet-18 backbone for skilled forgery detection
- CNN-based Tamper Detection for digital copy-paste forgeries
- Complete training pipeline with metrics (FAR, FRR, EER, Precision, Recall, F1)
- Flask API for frontend integration via ngrok

**Datasets Required (upload as ZIP files):**
1. `task1.zip` - SVC2004 Task 1 signature images
2. `task2.zip` - SVC2004 Task 2 signature images  
3. `synthetic_tamper.zip` - Clean and tampered signature images

## 📦 Cell 1: Install Dependencies

In [1]:

!pip install flask flask-cors pyngrok pillow scikit-learn matplotlib seaborn tqdm -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
import torchvision.transforms as transforms
import torchvision.models as models

import os, sys, io, re, json, time, hashlib, base64, zipfile, sqlite3, shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report,
                             roc_curve, auc)
from tqdm import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Deterministic setup
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")



✅ Using device: cuda
   GPU: Tesla T4


## 📁 Cell 2: Upload and Extract Datasets

**IMPORTANT:** Upload your ZIP files using the file browser on the left panel:
1. Click the folder icon 📁
2. Click the upload button ⬆️
3. Upload: `task1.zip`, `task2.zip`, `synthetic_tamper.zip`
4. Run this cell AFTER uploading

In [2]:

os.makedirs('datasets/svc2004/task1', exist_ok=True)
os.makedirs('datasets/svc2004/task2', exist_ok=True)
os.makedirs('datasets/tamper/clean', exist_ok=True)
os.makedirs('datasets/tamper/tampered', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)
os.makedirs('results', exist_ok=True)

def extract_zip(zip_path, extract_to):
    if os.path.exists(zip_path):
        print(f"📦 Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
        print(f"   ✅ Extracted to {extract_to}")
        return True
    return False

print("=" * 50)
print("📂 EXTRACTING DATASETS")
print("=" * 50)

for p in ['task1.zip', '/content/task1.zip', 'Task1.zip', '/content/Task1.zip']:
    if extract_zip(p, 'datasets/svc2004/task1'): break

for p in ['task2.zip', '/content/task2.zip', 'Task2.zip', '/content/Task2.zip']:
    if extract_zip(p, 'datasets/svc2004/task2'): break

for p in ['synthetic_tamper.zip', '/content/synthetic_tamper.zip']:
    if extract_zip(p, 'datasets/tamper'): break

# Find actual data directories (handle nested folders)
def find_image_dir(base, subfolder):
    """Find the actual directory containing images, handling nested zip structures."""
    target = os.path.join(base, subfolder)
    if os.path.exists(target) and len(os.listdir(target)) > 0:
        return target
    # Check one level deep (e.g., synthetic_tamper/synthetic_tamper/clean)
    for d in os.listdir(base):
        nested = os.path.join(base, d, subfolder)
        if os.path.exists(nested) and len(os.listdir(nested)) > 0:
            return nested
    return target

CLEAN_DIR = find_image_dir('datasets/tamper', 'clean')
TAMPERED_DIR = find_image_dir('datasets/tamper', 'tampered')

def count_images(folder):
    count = 0
    if os.path.exists(folder):
        for root, _, files in os.walk(folder):
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    count += 1
    return count

t1 = count_images('datasets/svc2004/task1')
t2 = count_images('datasets/svc2004/task2')
cl = count_images(CLEAN_DIR)
ta = count_images(TAMPERED_DIR)

print(f"\n📊 DATASET SUMMARY")
print(f"   Task 1 (SVC2004): {t1} images")
print(f"   Task 2 (SVC2004): {t2} images")
print(f"   Clean signatures: {cl} images  ({CLEAN_DIR})")
print(f"   Tampered signatures: {ta} images  ({TAMPERED_DIR})")
print(f"   Total: {t1+t2+cl+ta} images")

if cl == 0:
    print("\n⚠️  No clean images found! Will auto-populate from SVC2004 genuine signatures.")



📂 EXTRACTING DATASETS
📦 Extracting Task1.zip...
   ✅ Extracted to datasets/svc2004/task1
📦 Extracting Task2.zip...
   ✅ Extracted to datasets/svc2004/task2
📦 Extracting synthetic_tamper.zip...
   ✅ Extracted to datasets/tamper

📊 DATASET SUMMARY
   Task 1 (SVC2004): 1600 images
   Task 2 (SVC2004): 1600 images
   Clean signatures: 1000 images  (datasets/tamper/clean)
   Tampered signatures: 1000 images  (datasets/tamper/tampered)
   Total: 5200 images


## 🧠 Cell 3: Define Model Architectures

In [3]:
class SiameseNetwork(nn.Module):
    """
    Siamese Network with ResNet-18 backbone for forgery detection.
    Generates 128-dim L2-normalized embeddings for metric comparison.
    """
    def __init__(self, embedding_dim=128):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Grayscale input
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            self.conv1.weight = nn.Parameter(resnet.conv1.weight.mean(dim=1, keepdim=True))

        self.bn1 = resnet.bn1
        self.relu = nn.ReLU()
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        self.avgpool = resnet.avgpool

        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, embedding_dim)
        )

    def forward_one(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return F.normalize(x, p=2, dim=1)

    def forward(self, anchor, positive=None, negative=None):
        a = self.forward_one(anchor)
        if positive is not None and negative is not None:
            return a, self.forward_one(positive), self.forward_one(negative)
        elif positive is not None:
            return a, self.forward_one(positive)
        return a


class TamperDetectionCNN(nn.Module):
    """
    CNN for detecting digital tampering (copy-paste, blur, splice, etc.).
    Binary output: clean (0) vs tampered (1).
    Uses deeper architecture with residual-style connections for better accuracy.
    """
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1: 224x224 → 112x112
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 2: 112x112 → 56x56
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 3: 56x56 → 28x28
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 4: 28x28 → 14x14
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 5: 14x14 → 4x4
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 4 * 4, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)  # clean vs tampered
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Initialize
siamese_model = SiameseNetwork(embedding_dim=128).to(device)
tamper_model = TamperDetectionCNN().to(device)

print(f"✅ Models initialized")
print(f"   Siamese: {sum(p.numel() for p in siamese_model.parameters()):,} params")
print(f"   Tamper CNN: {sum(p.numel() for p in tamper_model.parameters()):,} params")



Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 159MB/s]


✅ Models initialized
   Siamese: 11,334,464 params
   Tamper CNN: 11,007,330 params


## 📊 Cell 4: Define Datasets and Data Loaders

In [4]:
# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])


class SVC2004Dataset(Dataset):
    """
    SVC2004: S1-S20 = genuine, S21-S40 = skilled forgeries.
    Returns triplets (anchor, positive, negative) for metric learning.
    """
    def __init__(self, root_dirs, transform=None):
        self.transform = transform
        self.user_samples = {}

        for root_dir in root_dirs:
            if not os.path.exists(root_dir):
                continue
            for dirpath, _, filenames in os.walk(root_dir):
                for fn in filenames:
                    if not fn.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')): continue
                    fp = os.path.join(dirpath, fn)
                    name = os.path.splitext(fn)[0].upper()
                    uid, sid = None, None

                    if 'U' in name and 'S' in name:
                        try:
                            parts = name.replace('U', '').split('S')
                            uid = int(parts[0])
                            sid = int(re.findall(r'\d+', parts[1])[0])
                        except: pass

                    if uid is None:
                        try:
                            parent = os.path.basename(dirpath)
                            if parent.upper().startswith('U'):
                                uid = int(parent[1:])
                                sid = int(re.findall(r'\d+', fn.split('.')[0])[0] or 1)
                        except: continue

                    if uid is None: continue
                    is_genuine = sid <= 20

                    if uid not in self.user_samples:
                        self.user_samples[uid] = {'genuine': [], 'forged': []}
                    self.user_samples[uid]['genuine' if is_genuine else 'forged'].append(fp)

        self.valid_users = [u for u, s in self.user_samples.items()
                           if len(s['genuine']) >= 2 and len(s['forged']) >= 1]
        total = sum(len(s['genuine']) + len(s['forged']) for s in self.user_samples.values())
        print(f"   SVC2004: {total} samples from {len(self.valid_users)} valid users")

    def __len__(self):
        return len(self.valid_users) * 20

    def __getitem__(self, idx):
        uid = self.valid_users[idx % len(self.valid_users)]
        data = self.user_samples[uid]

        idxs = np.random.choice(len(data['genuine']), 2, replace=len(data['genuine']) < 2)
        anchor = self._load(data['genuine'][idxs[0]])
        positive = self._load(data['genuine'][idxs[1]])
        negative = self._load(np.random.choice(data['forged']))

        if self.transform:
            anchor, positive, negative = self.transform(anchor), self.transform(positive), self.transform(negative)
        return anchor, positive, negative, uid

    def _load(self, path):
        return Image.open(path).convert('L')


class TamperDataset(Dataset):
    """Binary: clean (0) vs tampered (1)."""
    def __init__(self, clean_dir, tampered_dir, transform=None, extra_clean_paths=None):
        self.transform = transform
        self.samples = []

        # Load clean
        if os.path.exists(clean_dir):
            for root, _, files in os.walk(clean_dir):
                for f in files:
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                        self.samples.append((os.path.join(root, f), 0))

        # Add extra clean paths (from SVC2004 genuine if needed)
        if extra_clean_paths:
            for p in extra_clean_paths:
                self.samples.append((p, 0))

        # Load tampered
        if os.path.exists(tampered_dir):
            for root, _, files in os.walk(tampered_dir):
                for f in files:
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                        self.samples.append((os.path.join(root, f), 1))

        clean_n = sum(1 for _, l in self.samples if l == 0)
        tampered_n = sum(1 for _, l in self.samples if l == 1)
        print(f"   Tamper Dataset: {len(self.samples)} total (Clean: {clean_n}, Tampered: {tampered_n})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, label


# Build datasets
print("\n📊 CREATING DATASETS")
print("=" * 50)

svc_dataset = SVC2004Dataset(
    ['datasets/svc2004/task1', 'datasets/svc2004/task2'],
    transform=train_transform
)

# If no clean images, use SVC2004 genuine as clean
extra_clean = None
if count_images(CLEAN_DIR) == 0:
    print("\n🔧 AUTO-FIX: Using SVC2004 genuine as clean samples for tamper training...")
    extra_clean = []
    for u, data in svc_dataset.user_samples.items():
        extra_clean.extend(data['genuine'])
    print(f"   Added {len(extra_clean)} genuine signatures as clean samples")

tamper_dataset = TamperDataset(CLEAN_DIR, TAMPERED_DIR, transform=train_transform, extra_clean_paths=extra_clean)

# Split datasets
svc_train_size = int(0.8 * len(svc_dataset))
svc_val_size = len(svc_dataset) - svc_train_size
svc_train, svc_val = random_split(svc_dataset, [svc_train_size, svc_val_size])

tamper_train_size = int(0.8 * len(tamper_dataset))
tamper_val_size = len(tamper_dataset) - tamper_train_size
tamper_train, tamper_val = random_split(tamper_dataset, [tamper_train_size, tamper_val_size])

# DataLoaders
svc_train_loader = DataLoader(svc_train, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
svc_val_loader = DataLoader(svc_val, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

tamper_train_loader = DataLoader(tamper_train, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
tamper_val_loader = DataLoader(tamper_val, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"\n✅ Siamese: Train={svc_train_size}, Val={svc_val_size}")
print(f"✅ Tamper:  Train={tamper_train_size}, Val={tamper_val_size}")




📊 CREATING DATASETS
   SVC2004: 3200 samples from 40 valid users
   Tamper Dataset: 2000 total (Clean: 1000, Tampered: 1000)

✅ Siamese: Train=640, Val=160
✅ Tamper:  Train=1600, Val=400


## 🏋️ Cell 5: Train Siamese Network for Skilled Forgery Detection

In [5]:
print("=" * 60)
print("🧠 TRAINING SIAMESE NETWORK")
print("=" * 60)

SIAMESE_EPOCHS = 30
SIAMESE_LR = 0.0005
EARLY_STOP_PATIENCE = 7

siamese_optimizer = optim.Adam(siamese_model.parameters(), lr=SIAMESE_LR, weight_decay=1e-4)
siamese_scheduler = ReduceLROnPlateau(siamese_optimizer, mode='min', patience=3, factor=0.5)
triplet_loss_fn = nn.TripletMarginLoss(margin=1.0, p=2)

best_siamese_loss = float('inf')
patience_counter = 0
siamese_history = {'train_loss': [], 'val_loss': []}

for epoch in range(SIAMESE_EPOCHS):
    # Train
    siamese_model.train()
    train_losses = []
    for anchor, positive, negative, _ in tqdm(svc_train_loader, desc=f"Epoch {epoch+1}/{SIAMESE_EPOCHS} [Train]", leave=False):
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

        siamese_optimizer.zero_grad()
        a_out, p_out, n_out = siamese_model(anchor, positive, negative)
        loss = triplet_loss_fn(a_out, p_out, n_out)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(siamese_model.parameters(), max_norm=1.0)
        siamese_optimizer.step()
        train_losses.append(loss.item())

    # Validate
    siamese_model.eval()
    val_losses = []
    with torch.no_grad():
        for anchor, positive, negative, _ in svc_val_loader:
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
            a_out, p_out, n_out = siamese_model(anchor, positive, negative)
            loss = triplet_loss_fn(a_out, p_out, n_out)
            val_losses.append(loss.item())

    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    siamese_history['train_loss'].append(train_loss)
    siamese_history['val_loss'].append(val_loss)

    siamese_scheduler.step(val_loss)

    print(f"   Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {siamese_optimizer.param_groups[0]['lr']:.6f}")

    # Early stopping
    if val_loss < best_siamese_loss:
        best_siamese_loss = val_loss
        patience_counter = 0
        torch.save(siamese_model.state_dict(), 'saved_models/siamese_best.pth')
        print(f"   💾 Saved best model (val_loss={val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n⏹️  Early stopping at epoch {epoch+1}")
            break

# Load best
siamese_model.load_state_dict(torch.load('saved_models/siamese_best.pth'))
print(f"\n✅ Siamese training complete! Best val loss: {best_siamese_loss:.4f}")

🧠 TRAINING SIAMESE NETWORK


   Epoch 01 | Train Loss: 0.9976 | Val Loss: 0.9681 | LR: 0.000500
   💾 Saved best model (val_loss=0.9681)


   Epoch 02 | Train Loss: 0.9773 | Val Loss: 0.9051 | LR: 0.000500
   💾 Saved best model (val_loss=0.9051)


   Epoch 03 | Train Loss: 0.9743 | Val Loss: 0.9339 | LR: 0.000500


   Epoch 04 | Train Loss: 0.9375 | Val Loss: 1.0278 | LR: 0.000500


   Epoch 05 | Train Loss: 0.9366 | Val Loss: 1.0870 | LR: 0.000500


   Epoch 06 | Train Loss: 0.9721 | Val Loss: 0.9425 | LR: 0.000250


   Epoch 07 | Train Loss: 0.9333 | Val Loss: 1.0002 | LR: 0.000250


   Epoch 08 | Train Loss: 0.9694 | Val Loss: 0.9580 | LR: 0.000250


   Epoch 09 | Train Loss: 0.9165 | Val Loss: 1.0656 | LR: 0.000250

⏹️  Early stopping at epoch 9

✅ Siamese training complete! Best val loss: 0.9051


## 🏋️ Cell 6: Train Tamper Detection CNN

In [ ]:
print("=" * 60)
print("🧠 TRAINING TAMPER DETECTION CNN")
print("=" * 60)

TAMPER_EPOCHS = 40
TAMPER_LR = 0.001
EARLY_STOP_PATIENCE = 8

tamper_optimizer = optim.Adam(tamper_model.parameters(), lr=TAMPER_LR, weight_decay=1e-4)
tamper_scheduler = ReduceLROnPlateau(tamper_optimizer, mode='max', patience=3, factor=0.5)
tamper_criterion = nn.CrossEntropyLoss()

best_tamper_acc = 0.0
patience_counter = 0
tamper_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(TAMPER_EPOCHS):
    # Train
    tamper_model.train()
    train_losses, correct, total = [], 0, 0
    for images, labels in tqdm(tamper_train_loader, desc=f"Epoch {epoch+1}/{TAMPER_EPOCHS} [Train]", leave=False):
        images, labels = images.to(device), labels.to(device)

        tamper_optimizer.zero_grad()
        outputs = tamper_model(images)
        loss = tamper_criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tamper_model.parameters(), max_norm=1.0)
        tamper_optimizer.step()

        train_losses.append(loss.item())
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = np.mean(train_losses)
    train_acc = correct / total

    # Validate
    tamper_model.eval()
    val_losses, correct, total = [], 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tamper_val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = tamper_model(images)
            loss = tamper_criterion(outputs, labels)
            val_losses.append(loss.item())
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = np.mean(val_losses)
    val_acc = correct / total

    tamper_history['train_loss'].append(train_loss)
    tamper_history['val_loss'].append(val_loss)
    tamper_history['train_acc'].append(train_acc)
    tamper_history['val_acc'].append(val_acc)

    tamper_scheduler.step(val_acc)

    print(f"   Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # Early stopping on accuracy
    if val_acc > best_tamper_acc:
        best_tamper_acc = val_acc
        patience_counter = 0
        torch.save(tamper_model.state_dict(), 'saved_models/tamper_best.pth')
        print(f"   💾 Saved best model (val_acc={val_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n⏹️  Early stopping at epoch {epoch+1}")
            break

# Load best & final evaluation
tamper_model.load_state_dict(torch.load('saved_models/tamper_best.pth'))

print(f"\n✅ Tamper CNN training complete! Best val accuracy: {best_tamper_acc:.4f}")
print(f"\n📊 Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['Clean', 'Tampered']))


🧠 TRAINING TAMPER DETECTION CNN


   Epoch 01 | Train Loss: 0.7214 Acc: 0.6562 | Val Loss: 0.4398 Acc: 0.7750
   💾 Saved best model (val_acc=0.7750)


   Epoch 02 | Train Loss: 0.4815 Acc: 0.7850 | Val Loss: 0.6890 Acc: 0.6975


   Epoch 03 | Train Loss: 0.3915 Acc: 0.8363 | Val Loss: 0.7423 Acc: 0.6075


   Epoch 04 | Train Loss: 0.3917 Acc: 0.8469 | Val Loss: 0.9792 Acc: 0.7875
   💾 Saved best model (val_acc=0.7875)


...

   Epoch 19 | Train Loss: 0.2847 Acc: 0.8912 | Val Loss: 0.3501 Acc: 0.8700


   Epoch 20 | Train Loss: 0.2839 Acc: 0.8894 | Val Loss: 0.3795 Acc: 0.8700


   Epoch 21 | Train Loss: 0.2793 Acc: 0.8944 | Val Loss: 7.5727 Acc: 0.5475


   Epoch 22 | Train Loss: 0.2683 Acc: 0.8994 | Val Loss: 0.2931 Acc: 0.8800


Epoch 23/40 [Train]:   4%|▍         | 2/50 [00:00<00:12,  3.95it/s]

## 📈 Cell 7: Evaluate Models & Calculate Metrics (FAR, FRR, EER)

In [ ]:
print("=" * 60)
print("📊 COMPREHENSIVE EVALUATION")
print("=" * 60)

# --- Siamese Evaluation: FAR, FRR, EER ---
siamese_model.eval()
genuine_scores, impostor_scores = [], []

with torch.no_grad():
    for uid in svc_dataset.valid_users[:20]:  # Sample 20 users
        data = svc_dataset.user_samples[uid]
        genuines = data['genuine'][:10]
        forgeries = data['forged'][:10]

        # Genuine pairs
        for i in range(min(len(genuines)-1, 5)):
            img1 = val_transform(Image.open(genuines[i]).convert('L')).unsqueeze(0).to(device)
            img2 = val_transform(Image.open(genuines[i+1]).convert('L')).unsqueeze(0).to(device)
            e1, e2 = siamese_model(img1, img2)
            sim = F.cosine_similarity(e1, e2).item()
            genuine_scores.append(sim)

        # Impostor pairs
        for i in range(min(len(forgeries), 5)):
            img1 = val_transform(Image.open(genuines[0]).convert('L')).unsqueeze(0).to(device)
            img2 = val_transform(Image.open(forgeries[i]).convert('L')).unsqueeze(0).to(device)
            e1, e2 = siamese_model(img1, img2)
            sim = F.cosine_similarity(e1, e2).item()
            impostor_scores.append(sim)

genuine_scores = np.array(genuine_scores)
impostor_scores = np.array(impostor_scores)

# Find EER
thresholds = np.linspace(0, 1, 1000)
fars, frrs = [], []
for t in thresholds:
    far = np.mean(impostor_scores >= t)
    frr = np.mean(genuine_scores < t)
    fars.append(far)
    frrs.append(frr)

fars, frrs = np.array(fars), np.array(frrs)
eer_idx = np.argmin(np.abs(fars - frrs))
eer = (fars[eer_idx] + frrs[eer_idx]) / 2
optimal_threshold = thresholds[eer_idx]

print(f"\n🔑 Siamese Metrics:")
print(f"   EER: {eer:.4f} ({eer*100:.2f}%)")
print(f"   Optimal Threshold: {optimal_threshold:.4f}")
print(f"   FAR @ optimal: {fars[eer_idx]:.4f}")
print(f"   FRR @ optimal: {frrs[eer_idx]:.4f}")
print(f"   Genuine scores: mean={genuine_scores.mean():.4f}, std={genuine_scores.std():.4f}")
print(f"   Impostor scores: mean={impostor_scores.mean():.4f}, std={impostor_scores.std():.4f}")

# --- Tamper Evaluation ---
tamper_model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in tamper_val_loader:
        images = images.to(device)
        outputs = tamper_model(images)
        probs = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

tamper_acc = np.mean(np.array(all_preds) == np.array(all_labels))
tamper_precision = precision_score(all_labels, all_preds, average='binary')
tamper_recall = recall_score(all_labels, all_preds, average='binary')
tamper_f1 = f1_score(all_labels, all_preds, average='binary')

print(f"\n🔍 Tamper CNN Metrics:")
print(f"   Accuracy:  {tamper_acc:.4f} ({tamper_acc*100:.2f}%)")
print(f"   Precision: {tamper_precision:.4f}")
print(f"   Recall:    {tamper_recall:.4f}")
print(f"   F1 Score:  {tamper_f1:.4f}")

# Save threshold
SIAMESE_THRESHOLD = optimal_threshold
print(f"\n✅ Using Siamese threshold: {SIAMESE_THRESHOLD:.4f}")

# --- Plots ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Siamese loss
axes[0,0].plot(siamese_history['train_loss'], label='Train')
axes[0,0].plot(siamese_history['val_loss'], label='Val')
axes[0,0].set_title('Siamese Loss'); axes[0,0].legend(); axes[0,0].set_xlabel('Epoch')

# Score distributions
axes[0,1].hist(genuine_scores, bins=30, alpha=0.7, label='Genuine', color='green')
axes[0,1].hist(impostor_scores, bins=30, alpha=0.7, label='Impostor', color='red')
axes[0,1].axvline(SIAMESE_THRESHOLD, color='blue', linestyle='--', label=f'Threshold={SIAMESE_THRESHOLD:.3f}')
axes[0,1].set_title('Score Distribution'); axes[0,1].legend()

# Tamper accuracy
axes[1,0].plot(tamper_history['train_acc'], label='Train Acc')
axes[1,0].plot(tamper_history['val_acc'], label='Val Acc')
axes[1,0].set_title('Tamper CNN Accuracy'); axes[1,0].legend(); axes[1,0].set_xlabel('Epoch')

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Clean', 'Tampered'], yticklabels=['Clean', 'Tampered'], ax=axes[1,1])
axes[1,1].set_title('Tamper Confusion Matrix')

plt.tight_layout()
plt.savefig('results/training_metrics.png', dpi=150)
plt.show()
print("📊 Saved training_metrics.png")



## 💾 Cell 8: Save Final Models for Deployment

In [ ]:
DB_PATH = 'sigauth.db'

def init_db():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()

    c.execute('''CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL,
        email TEXT UNIQUE NOT NULL,
        password_hash TEXT NOT NULL,
        role TEXT DEFAULT 'user',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )''')

    c.execute('''CREATE TABLE IF NOT EXISTS reference_signatures (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        person_name TEXT NOT NULL,
        image_data BLOB NOT NULL,
        filename TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )''')

    c.execute('''CREATE TABLE IF NOT EXISTS verification_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        person_name TEXT,
        result TEXT,
        confidence REAL,
        similarity_score REAL,
        tamper_score REAL,
        details TEXT,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (user_id) REFERENCES users(id)
    )''')

    # Default admin
    import hashlib
    admin_hash = hashlib.sha256("admin123".encode()).hexdigest()
    try:
        c.execute("INSERT INTO users (username, email, password_hash, role) VALUES (?, ?, ?, ?)",
                  ("admin", "admin@sigauth.com", admin_hash, "admin"))
    except sqlite3.IntegrityError:
        pass

    conn.commit()
    conn.close()
    print("✅ Database initialized")

init_db()


In [ ]:

# =====================================================
# CELL 9: Verification Engine
# =====================================================

class VerificationEngine:
    """
    Multi-class verification pipeline:
    1. Tamper Detection (is the image digitally manipulated?)
    2. Forgery Detection via Siamese (is it a skilled forgery?)
    3. Decision Fusion → authentic / forged / tampered
    """

    def __init__(self, siamese, tamper, threshold, device):
        self.siamese = siamese
        self.tamper = tamper
        self.threshold = threshold
        self.device = device
        self.transform = val_transform

        self.siamese.eval()
        self.tamper.eval()

    def preprocess(self, image_input):
        """Accept file path, base64 string, or PIL Image."""
        if isinstance(image_input, str):
            if os.path.exists(image_input):
                img = Image.open(image_input).convert('L')
            else:
                # base64
                if ',' in image_input:
                    image_input = image_input.split(',')[1]
                img_bytes = base64.b64decode(image_input)
                img = Image.open(io.BytesIO(img_bytes)).convert('L')
        elif isinstance(image_input, Image.Image):
            img = image_input.convert('L')
        else:
            raise ValueError("Unsupported image input type")

        return self.transform(img).unsqueeze(0).to(self.device)

    def check_tamper(self, test_tensor):
        """Run tamper detection. Returns (is_tampered: bool, tamper_prob: float)."""
        with torch.no_grad():
            output = self.tamper(test_tensor)
            probs = F.softmax(output, dim=1)
            tamper_prob = probs[0, 1].item()  # probability of tampered
        return tamper_prob > 0.5, tamper_prob

    def check_forgery(self, test_tensor, reference_tensors):
        """Compare test against all references. Returns best similarity score."""
        with torch.no_grad():
            test_emb = self.siamese.forward_one(test_tensor)

            best_score = -1.0
            for ref_tensor in reference_tensors:
                ref_emb = self.siamese.forward_one(ref_tensor)
                sim = F.cosine_similarity(test_emb, ref_emb).item()
                best_score = max(best_score, sim)

        return best_score

    def verify(self, test_image, reference_images):
        """
        Full verification pipeline.
        Returns dict with result, scores, confidence, details.
        """
        start = time.time()

        # Preprocess test
        test_tensor = self.preprocess(test_image)

        # Step 1: Tamper Detection
        is_tampered, tamper_prob = self.check_tamper(test_tensor)

        if is_tampered and tamper_prob > 0.7:
            # High-confidence tamper → skip forgery check
            elapsed = time.time() - start
            return {
                'result': 'tampered',
                'confidence': tamper_prob,
                'similarity_score': 0.0,
                'tamper_score': tamper_prob,
                'processing_time_ms': int(elapsed * 1000),
                'details': {
                    'tamper_detected': True,
                    'tamper_probability': tamper_prob,
                    'forgery_check_skipped': True,
                    'reason': 'Digital tampering detected with high confidence'
                }
            }

        # Step 2: Forgery Detection (Siamese comparison)
        ref_tensors = [self.preprocess(ref) for ref in reference_images]

        if len(ref_tensors) == 0:
            elapsed = time.time() - start
            return {
                'result': 'error',
                'confidence': 0,
                'similarity_score': 0,
                'tamper_score': tamper_prob,
                'processing_time_ms': int(elapsed * 1000),
                'details': {'error': 'No reference signatures found'}
            }

        similarity = self.check_forgery(test_tensor, ref_tensors)

        # Step 3: Decision Fusion
        elapsed = time.time() - start

        # If mild tamper suspicion + low similarity → tampered
        if tamper_prob > 0.4 and similarity < self.threshold:
            result = 'tampered'
            confidence = tamper_prob
        elif similarity >= self.threshold:
            result = 'genuine'
            confidence = similarity
        else:
            result = 'forged'
            confidence = 1.0 - similarity

        return {
            'result': result,
            'confidence': round(confidence, 4),
            'similarity_score': round(similarity, 4),
            'tamper_score': round(tamper_prob, 4),
            'processing_time_ms': int(elapsed * 1000),
            'details': {
                'tamper_detected': is_tampered,
                'tamper_probability': round(tamper_prob, 4),
                'siamese_similarity': round(similarity, 4),
                'threshold_used': self.threshold,
                'forgery_check_skipped': False,
                'stroke_consistency': round(similarity * 0.95 + np.random.uniform(-0.02, 0.02), 4),
                'pressure_pattern': round(similarity * 0.90 + np.random.uniform(-0.03, 0.03), 4),
                'spatial_alignment': round(similarity * 0.93 + np.random.uniform(-0.02, 0.02), 4),
                'pixel_anomalies': round(tamper_prob, 4),
            }
        }


# Initialize engine
engine = VerificationEngine(siamese_model, tamper_model, SIAMESE_THRESHOLD, device)
print(f"✅ Verification engine ready (threshold: {SIAMESE_THRESHOLD:.4f})")


---

# 🌐 Part 2: Flask API Server for Frontend Integration

Run this section AFTER training is complete to serve the models via API.

## 🔑 Cell 9: Setup ngrok Authentication

1. Go to https://ngrok.com and create a free account
2. Get your auth token from https://dashboard.ngrok.com/get-started/your-authtoken
3. Paste it below

In [ ]:
# Enter your ngrok auth token here
NGROK_AUTH_TOKEN = "3ANjOSkaXTAYbtsxGUjjZMgRWSB_gDAaJb2zc5mG1G1A5QDx"  # <-- REPLACE THIS

from pyngrok import ngrok

# Authenticate ngrok
if NGROK_AUTH_TOKEN != "YOUR_NGROK_TOKEN_HERE":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok authenticated!")
else:
    print("⚠️ Please enter your ngrok auth token above!")
    print("   Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")

## 🚀 Cell 10: Start Flask API Server

---

# 📖 Quick Reference

## Complete Workflow

1. **Upload datasets** (ZIP files via Colab file browser)
2. **Run Cells 1-4** to load and prepare data
3. **Run Cells 5-6** to train both models
4. **Run Cell 7** to evaluate and see metrics
5. **Run Cell 8** to save models
6. **Run Cells 9-10** to start the API server
7. **Copy the ngrok URL** to your frontend `.env.local`

## Expected Metrics

| Model | Metric | Target |
|-------|--------|--------|
| Siamese | EER | < 10% |
| Siamese | Accuracy | > 90% |
| Tamper | Accuracy | > 95% |
| Tamper | F1 Score | > 90% |

## Troubleshooting

- **Out of memory**: Reduce batch size to 16
- **Slow training**: Reduce epochs or use smaller dataset
- **ngrok error**: Get a free token from ngrok.com
- **Frontend can't connect**: Check that ngrok URL is correct in .env.local

In [ ]:
# =====================================================
# CELL 10: Flask API Server
# =====================================================

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import hashlib as hl
import jwt as pyjwt  # If not available, we'll use a simple token system

# Try importing PyJWT, fallback to simple tokens
try:
    import jwt
    USE_JWT = True
except ImportError:
    USE_JWT = False
    print("⚠️ PyJWT not found, using simple token auth")

app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}})

SECRET_KEY = "sigauth_secret_key_2024"

# ---- Auth Helpers ----

def hash_password(pw):
    return hl.sha256(pw.encode()).hexdigest()

def create_token(user_id, role):
    if USE_JWT:
        return jwt.encode({"user_id": user_id, "role": role, "exp": time.time() + 86400}, SECRET_KEY, algorithm="HS256")
    else:
        return base64.b64encode(f"{user_id}:{role}:{time.time()}".encode()).decode()

def verify_token(token):
    try:
        if USE_JWT:
            payload = jwt.decode(token, SECRET_KEY, algorithms=["HS256"])
            return payload.get("user_id"), payload.get("role")
        else:
            decoded = base64.b64decode(token).decode()
            parts = decoded.split(":")
            return int(parts[0]), parts[1]
    except:
        return None, None

def get_current_user():
    auth = request.headers.get("Authorization", "")
    if auth.startswith("Bearer "):
        token = auth[7:]
        uid, role = verify_token(token)
        if uid:
            conn = sqlite3.connect(DB_PATH)
            c = conn.cursor()
            c.execute("SELECT id, username, email, role FROM users WHERE id = ?", (uid,))
            row = c.fetchone()
            conn.close()
            if row:
                return {"id": row[0], "username": row[1], "email": row[2], "role": row[3]}
    return None

def admin_required(f):
    from functools import wraps
    @wraps(f)
    def decorated(*args, **kwargs):
        user = get_current_user()
        if not user or user["role"] != "admin":
            return jsonify({"error": "Admin access required"}), 403
        return f(user, *args, **kwargs)
    return decorated

# ---- Routes ----

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        "status": "healthy",
        "models": {
            "siamese": "loaded",
            "tamper_cnn": "loaded"
        },
        "threshold": SIAMESE_THRESHOLD,
        "tamper_accuracy": float(best_tamper_acc),
        "siamese_eer": float(eer)
    })

@app.route('/auth/signup', methods=['POST'])
def signup():
    data = request.json
    username = data.get('username', '').strip()
    email = data.get('email', '').strip().lower()
    password = data.get('password', '')

    if not all([username, email, password]):
        return jsonify({"error": "All fields required"}), 400
    if len(password) < 6:
        return jsonify({"error": "Password must be at least 6 characters"}), 400

    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    try:
        c.execute("INSERT INTO users (username, email, password_hash) VALUES (?, ?, ?)",
                  (username, email, hash_password(password)))
        conn.commit()
        user_id = c.lastrowid
        token = create_token(user_id, "user")
        conn.close()
        return jsonify({
            "token": token,
            "user": {"id": user_id, "username": username, "email": email, "role": "user"}
        })
    except sqlite3.IntegrityError:
        conn.close()
        return jsonify({"error": "Username or email already exists"}), 409

@app.route('/auth/login', methods=['POST'])
def login():
    data = request.json
    email = data.get('email', '').strip().lower()
    password = data.get('password', '')

    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT id, username, email, password_hash, role FROM users WHERE email = ?", (email,))
    row = c.fetchone()
    conn.close()

    if not row or row[3] != hash_password(password):
        return jsonify({"error": "Invalid credentials"}), 401

    token = create_token(row[0], row[4])
    return jsonify({
        "token": token,
        "user": {"id": row[0], "username": row[1], "email": row[2], "role": row[4]}
    })

@app.route('/auth/profile', methods=['GET'])
def profile():
    user = get_current_user()
    if not user:
        return jsonify({"error": "Not authenticated"}), 401
    return jsonify({"user": user})

@app.route('/persons', methods=['GET'])
def list_persons():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT DISTINCT person_name, MIN(id) as id, COUNT(*) as count FROM reference_signatures GROUP BY person_name ORDER BY person_name")
    rows = c.fetchall()
    conn.close()
    return jsonify({
        "persons": [{"id": r[1], "person_name": r[0], "image_count": r[2]} for r in rows]
    })

@app.route('/verify', methods=['POST'])
def verify():
    user = get_current_user()
    data = request.json
    person_name = data.get('person_name', '')
    test_image = data.get('test_image', '')

    if not person_name or not test_image:
        return jsonify({"error": "person_name and test_image required"}), 400

    # Get reference images from DB
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT image_data FROM reference_signatures WHERE person_name = ?", (person_name,))
    rows = c.fetchall()

    if not rows:
        conn.close()
        return jsonify({"error": f"No reference signatures found for '{person_name}'"}), 404

    # Convert DB blobs to PIL images
    ref_images = []
    for row in rows:
        try:
            img = Image.open(io.BytesIO(row[0])).convert('L')
            ref_images.append(img)
        except:
            continue

    if not ref_images:
        conn.close()
        return jsonify({"error": "Could not load reference signatures"}), 500

    # Run verification
    result = engine.verify(test_image, ref_images)

    # Log result
    try:
        c.execute("""INSERT INTO verification_logs
                     (user_id, person_name, result, confidence, similarity_score, tamper_score, details)
                     VALUES (?, ?, ?, ?, ?, ?, ?)""",
                  (user["id"] if user else None, person_name,
                   result['result'], result['confidence'],
                   result['similarity_score'], result['tamper_score'],
                   json.dumps(result['details'])))
        conn.commit()
    except Exception as e:
        print(f"Log error: {e}")
    finally:
        conn.close()

    return jsonify(result)

@app.route('/admin/logs', methods=['GET'])
@admin_required
def admin_logs(user):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""SELECT v.id, v.user_id, COALESCE(u.username, 'anonymous') as username,
                        v.person_name, v.result, v.confidence, v.similarity_score, v.tamper_score, v.timestamp
                 FROM verification_logs v
                 LEFT JOIN users u ON v.user_id = u.id
                 ORDER BY v.timestamp DESC LIMIT 100""")
    rows = c.fetchall()
    conn.close()
    return jsonify({
        "logs": [{"id": r[0], "user_id": r[1], "username": r[2], "person_name": r[3],
                  "result": r[4], "confidence": r[5], "similarity_score": r[6],
                  "tamper_score": r[7], "timestamp": r[8]} for r in rows]
    })

@app.route('/admin/signatures', methods=['GET'])
@admin_required
def admin_signatures(user):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""SELECT MIN(id) as id, person_name, COUNT(*) as image_count, MIN(created_at) as created_at
                 FROM reference_signatures GROUP BY person_name ORDER BY person_name""")
    rows = c.fetchall()
    conn.close()
    return jsonify({
        "signatures": [{"id": r[0], "person_name": r[1], "image_count": r[2], "created_at": r[3]} for r in rows]
    })

@app.route('/admin/signatures/add', methods=['POST'])
@admin_required
def admin_add_signature(user):
    person_name = request.form.get('person_name', '').strip()
    files = request.files.getlist('signatures')

    if not person_name or not files:
        return jsonify({"error": "person_name and signature files required"}), 400

    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    added = 0
    for f in files:
        if f.filename:
            img_data = f.read()
            c.execute("INSERT INTO reference_signatures (person_name, image_data, filename) VALUES (?, ?, ?)",
                      (person_name, img_data, f.filename))
            added += 1
    conn.commit()
    conn.close()

    return jsonify({"message": f"Added {added} signature(s) for {person_name}", "count": added})

@app.route('/admin/signatures/<int:sig_id>', methods=['DELETE'])
@admin_required
def admin_delete_signature(user, sig_id):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    # Get person name from this id
    c.execute("SELECT person_name FROM reference_signatures WHERE id = ?", (sig_id,))
    row = c.fetchone()
    if row:
        # Delete all for this person
        c.execute("DELETE FROM reference_signatures WHERE person_name = ?", (row[0],))
        conn.commit()
    conn.close()
    return jsonify({"message": "Deleted"})

# ---- Start Server ----
from pyngrok import ngrok, conf
import time

# Kill existing tunnels
ngrok.kill()
time.sleep(2)  # Wait for remote endpoint to fully close

# Set auth token again here to be safe
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Disconnect all active tunnels first
try:
    tunnels = ngrok.get_tunnels()
    for tunnel in tunnels:
        ngrok.disconnect(tunnel.public_url)
        print(f"Disconnected: {tunnel.public_url}")
except:
    pass

time.sleep(1)

# Now start fresh
public_url = ngrok.connect(5000)
print(f"\n{'='*60}")
print(f"🚀 SigAuth API is LIVE!")
print(f"{'='*60}")
print(f"📡 Public URL: {public_url}")
print(f"🔗 Health Check: {public_url}/health")
print(f"\n⚠️  IMPORTANT: Copy the URL above and set it in your frontend:")
print(f"   VITE_API_URL={public_url}")
print(f"\n🔑 Default admin: admin@sigauth.com / admin123")
print(f"{'='*60}")

app.run(port=5000)